# Predicting Bitcoin Prices from Chainlet Data, Explained with SHAP

**CAP 5619, AI for Finance. Alex Hubenthal.**

This notebook predicts the daily Bitcoin price for every day of December 2017, one day ahead, from
lagged prices and chainlet features, and explains the model with SHAP. The data are the three files
provided with the course: the occurrence chainlets, the amount chainlets and the daily price.

Constraints the notebook is built around:

* Chainlet features come from both the occurrence file and the amount file.
* RMSE is computed in USD over the 31 days of December 2017.
* Dates are parsed with datetime functions and never sliced as strings. `year`, `day` and `date` are
  never model inputs.
* The horizon is one day, and nothing from December 2017 reaches training, scaling or model selection.
* Regression or feedforward networks only, no ensembles.
* SHAP explains at least three December predictions, each followed by a written reading.

## 0. Setup

The first cell checks the Python version and installs any missing package, so the notebook can start on
a machine with nothing installed. The second cell defines the constants once, so the test window and the
horizon cannot drift between cells.

In [1]:
import importlib
import importlib.util
import subprocess
import sys

MIN_PYTHON = (3, 9)
REQUIRED = {  # import name -> pip name
    "numpy": "numpy",
    "pandas": "pandas",
    "sklearn": "scikit-learn",
    "matplotlib": "matplotlib",
    "shap": "shap",
}

if sys.version_info < MIN_PYTHON:
    raise RuntimeError(
        "Python %d.%d or newer is required, found %d.%d. Install a newer Python and rerun."
        % (MIN_PYTHON + tuple(sys.version_info[:2]))
    )

missing = [pip for mod, pip in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if missing:
    result = subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "--disable-pip-version-check", *missing],
        capture_output=True,
        text=True,
    )
    if result.returncode != 0:
        # An "externally managed environment" error means the notebook is running in the system Python.
        raise RuntimeError(
            "pip could not install %s. If it reports an externally managed environment, run the notebook "
            "from a virtual environment (python -m venv .venv).\n%s" % (", ".join(missing), result.stderr[-1500:])
        )
    importlib.invalidate_caches()

versions = {name: importlib.import_module(name).__version__ for name in REQUIRED}
print("Python %d.%d.%d | " % tuple(sys.version_info[:3]) + ", ".join("%s %s" % item for item in versions.items()))

Python 3.13.18 | numpy 2.5.5, pandas 3.1.1, sklearn 1.9.1, matplotlib 3.11.2, shap 0.54.1


In [2]:
import os
import urllib.request

import numpy as np
import pandas as pd

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 12)

SEED = 5619
np.random.seed(SEED)

# The three course files are read from disk and never embedded in the notebook.
DATA_DIR = "data"
FILES = {
    "occ": "OccChainletsInTime.txt",
    "amo": "AmoChainletsInTime.txt",
    "price": "pricedBitcoin2009-2018.csv",
}
GITHUB_RAW = "https://raw.githubusercontent.com/cakcora/BitcoinChainnet/master/data/"

# The prediction task, fixed once. December 2017 is the test period; September to November 2017 is the
# validation period used for every modelling choice.
HORIZON_DAYS = 1
TEST_START = pd.Timestamp("2017-12-01")
TEST_END = pd.Timestamp("2017-12-31")
VAL_START = pd.Timestamp("2017-09-01")
VAL_END = pd.Timestamp("2017-11-30")

# The chainlet matrix is 20 x 20; row and column 20 hold the capped cells.
N_CHAINLET = 20


def ensure_data_files(data_dir=DATA_DIR):
    """Return the paths of the three course files, downloading any that are missing."""
    os.makedirs(data_dir, exist_ok=True)
    paths = {}
    for key, name in FILES.items():
        path = os.path.join(data_dir, name)
        if not os.path.exists(path):
            # Download under a temporary name so an interrupted run never leaves a half-written file.
            partial = path + ".part"
            urllib.request.urlretrieve(GITHUB_RAW + name, partial)
            os.replace(partial, path)
        paths[key] = path
        print("%-5s %s  %.1f MB" % (key, path, os.path.getsize(path) / 1e6))
    return paths


PATHS = ensure_data_files()

occ   data/OccChainletsInTime.txt  3.0 MB
amo   data/AmoChainletsInTime.txt  8.0 MB
price data/pricedBitcoin2009-2018.csv  0.1 MB


## 1. Load data and parse dates

The price file's `date` column is parsed once with `pd.to_datetime` and an explicit month/day/year
format. The chainlet files carry only `year` and `day` (the day of the year), so their dates are
assembled from those two numbers with `pd.to_datetime` and a timedelta, with no string handling. Each
loader asserts what it assumes and stops if something is off. The parsed dates are checked against
`year` and `day`, which are then dropped so they cannot reach a model by accident.

In [3]:
def load_price(path):
    """Read the price file and return a frame indexed by date with columns price and totaltx."""
    raw = pd.read_csv(path)
    assert list(raw.columns) == ["date", "price", "year", "day", "totaltx"], "unexpected price columns"

    # An explicit format avoids guessing the day/month order. The year and day checks below confirm it.
    date = pd.to_datetime(raw["date"], format="%m/%d/%Y %H:%M")
    assert date.notna().all()
    assert (date == date.dt.normalize()).all(), "price dates are not all at midnight"
    assert date.is_unique and date.is_monotonic_increasing
    assert (date.dt.year == raw["year"]).all()
    assert (date.dt.day_of_year == raw["day"]).all()
    assert (raw["price"] > 0).all()

    return raw[["price", "totaltx"]].set_index(pd.DatetimeIndex(date, name="date"))


def load_chainlets(path, prefix):
    """Read one chainlet file and return a frame indexed by date.

    Columns are `<prefix>_total` and the 400 cells `<prefix>_<i>_<o>` of the 20 x 20 chainlet matrix.
    """
    raw = pd.read_csv(path, sep="\t")
    cells = ["%d:%d" % (i, o) for i in range(1, N_CHAINLET + 1) for o in range(1, N_CHAINLET + 1)]
    assert list(raw.columns) == ["year", "day", "totaltx"] + cells, "unexpected chainlet columns"
    assert raw[cells].notna().all().all() and (raw[cells] >= 0).all().all()
    # The third column is compared with the cells instead of being trusted by its name.
    assert (raw[cells].sum(axis=1) == raw["totaltx"]).all(), "totaltx is not the sum of the cells"

    # January 1 of `year` plus (day - 1) days. A day number past the end of its year would roll into the
    # next year, which the year check catches.
    january_first = pd.to_datetime(pd.DataFrame({"year": raw["year"], "month": 1, "day": 1}))
    date = january_first + pd.to_timedelta(raw["day"] - 1, unit="D")
    assert date.is_unique and date.is_monotonic_increasing
    assert (date.dt.year == raw["year"]).all()
    assert (date.dt.day_of_year == raw["day"]).all()

    cell_frame = raw[cells].rename(columns=lambda cell: "%s_%s" % (prefix, cell.replace(":", "_")))
    # One concat instead of inserting into a 400-column frame, which pandas flags as fragmented.
    frame = pd.concat([raw["totaltx"].rename("%s_total" % prefix), cell_frame], axis=1)
    frame.index = pd.DatetimeIndex(date, name="date")
    return frame

In [4]:
price = load_price(PATHS["price"])
occ = load_chainlets(PATHS["occ"], "occ")
amo = load_chainlets(PATHS["amo"], "amo")

# December 2017 is read in full: scoring needs the actual prices, and each prediction uses the previous
# day's price.
test_days = pd.date_range(TEST_START, TEST_END, freq="D")
assert len(test_days) == 31 and test_days.isin(price.index).all()

# The days the price file lacks do exist in both chainlet files. Section 2 decides what to do with them.
price_gaps = pd.date_range(price.index.min(), price.index.max(), freq="D").difference(price.index)
assert price_gaps.isin(occ.index).all() and price_gaps.isin(amo.index).all()


def describe_source(name, frame):
    calendar = pd.date_range(frame.index.min(), frame.index.max(), freq="D")
    gaps = calendar.difference(frame.index)
    return {
        "source": name,
        "rows": len(frame),
        "first day": frame.index.min().date(),
        "last day": frame.index.max().date(),
        "index dtype": str(frame.index.dtype),
        "missing days": ", ".join(day.date().isoformat() for day in gaps) or "none",
    }


pd.DataFrame(
    [describe_source("price", price), describe_source("occurrence", occ), describe_source("amount", amo)]
).set_index("source")

,rows,first day,last day,index dtype,missing days
source,,,,,
price,2404,2011-06-01,2017-12-31,datetime64[us],"2012-12-31, 2016-12-31"
occurrence,3281,2009-01-03,2017-12-31,datetime64[us],"2009-01-04, 2009-01-05, 2009-01-06, 2009-01-07"
amount,3281,2009-01-03,2017-12-31,datetime64[us],"2009-01-04, 2009-01-05, 2009-01-06, 2009-01-07"


In [5]:
from IPython.display import display

display(price.head(3))
display(occ.iloc[:3, :4])
display(amo.iloc[:3, :4])

,price,totaltx
date,,
2011-06-01,9.57,6464
2011-06-02,10.60,7632
2011-06-03,14.29,8502


,occ_total,occ_1_1,occ_1_2,occ_1_3
date,,,,
2009-01-03,1,1,0,0
2009-01-08,14,14,0,0
2009-01-09,7,7,0,0


,amo_total,amo_1_1,amo_1_2,amo_1_3
date,,,,
2009-01-03,5000000000,5000000000,0,0
2009-01-08,70000000000,70000000000,0,0
2009-01-09,35000000000,35000000000,0,0


What the checks found, before anything is built on these frames:

* The price file runs from 2011-06-01 to 2017-12-31 (2,404 days), not from 2009 to 2018 as its name
  suggests. It lacks two calendar days, 2012-12-31 and 2016-12-31, both day 366 of a leap year, which is
  what a loop over days 1 to 365 would skip. The chainlet files do have those two days, so the merge in
  Section 2 has to decide what to do with them.
* The chainlet files run from 2009-01-03 to 2017-12-31 (3,281 days) and miss only four days at the very
  start, 2009-01-04 to 2009-01-07. Neither file reaches 2018.
* In both chainlet files the third column, `totaltx`, equals the sum of the 400 cells. In the
  occurrence file it is a transaction count. In the amount file it is a total in satoshis: the first row
  is 5,000,000,000, which is 50 bitcoin and the reward of the genesis block, where the occurrence file
  shows one transaction. The two columns share a name but not a meaning, so they are renamed `occ_total`
  and `amo_total`.
* December 2017 is complete in the price file (31 days). It is read because scoring needs the actual
  prices. It is not used for fitting, scaling or selection.

## 2. Merge occurrence, amount and price

The three frames are joined on a complete daily calendar from the first to the last price date, so every
calendar day is a row and a lag of one row is a lag of one day. Joining on the price dates alone would
skip the two missing days, and a one-row lag across such a gap would span two days.

Those two days keep their chainlet rows and get an empty price. They are not filled forward: a filled
price is an observation nobody made, and the filled day would carry a return of exactly zero. Rows that
need one of these prices, as a lag or as a target, are dropped when the model table is built in Section 3.

Nothing in this section is fitted or estimated, so keeping December 2017 in the frame does not let it
reach training.

In [6]:
def merge_sources(price, occ, amo):
    """Join the three frames on a complete daily calendar over the price period."""
    calendar = pd.date_range(price.index.min(), price.index.max(), freq="D")
    assert occ.index.equals(amo.index), "occurrence and amount files cover different days"
    assert calendar.isin(occ.index).all(), "a chainlet day is missing inside the price period"
    # Reindexing the price onto the calendar keeps its two missing days as empty rows instead of skipping them.
    return pd.concat([price["price"].reindex(calendar), occ.reindex(calendar), amo.reindex(calendar)], axis=1)


data = merge_sources(price, occ, amo)
assert data.index.is_unique and data.index.is_monotonic_increasing

# The price file's transaction count is the occurrence total under another name. Equality on every day shows
# that the three files are aligned on the same calendar date.
count_mismatches = int((price["totaltx"] != data.loc[price.index, "occ_total"]).sum())
assert count_mismatches == 0, "transaction counts disagree, so the dates do not line up"

OCC_CELLS = [name for name in data.columns if name.startswith("occ_") and name != "occ_total"]
AMO_CELLS = [name for name in data.columns if name.startswith("amo_") and name != "amo_total"]
assert len(OCC_CELLS) == len(AMO_CELLS) == N_CHAINLET ** 2

# Only the price may be empty, and only on the days the price file lacks.
assert data.drop(columns="price").notna().all().all()
assert data.index[data["price"].isna()].equals(price_gaps)

# Every score rests on the validation and test months, so they must be complete in every column.
evaluation_days = pd.date_range(VAL_START, TEST_END, freq="D")
assert data.loc[evaluation_days].notna().all().all(), "gaps in the validation or test period"

pd.Series(
    {
        "calendar days": len(data),
        "first day": data.index.min().date(),
        "last day": data.index.max().date(),
        "days with a price": int(data["price"].notna().sum()),
        "days without a price": ", ".join(day.date().isoformat() for day in price_gaps),
        "transaction count mismatches": count_mismatches,
        "occurrence cells": len(OCC_CELLS),
        "amount cells": len(AMO_CELLS),
        "columns": data.shape[1],
    },
    name="value",
).to_frame()

,value
calendar days,2406
first day,2011-06-01
last day,2017-12-31
days with a price,2404
days without a price,"2012-12-31, 2016-12-31"
transaction count mismatches,0
occurrence cells,400
amount cells,400
columns,803


In [7]:
# The gap days keep their chainlet rows and have an empty price.
around_gaps = pd.DatetimeIndex([day + pd.Timedelta(days=offset) for day in price_gaps for offset in (-1, 0, 1)])
data.loc[around_gaps, ["price", "occ_total", "amo_total"]]

,price,occ_total,amo_total
2012-12-30,13.45,43472,86628344663514
2012-12-31,NaN,27844,136824068787140
2013-01-01,13.30,35315,189995377467014
2016-12-30,960.47,288346,207216638799652
2016-12-31,NaN,224275,119964952102904
2017-01-01,997.69,204354,121705100814236


What the merge shows:

* The merged frame has 2,406 rows, one per calendar day from 2011-06-01 to 2017-12-31, and 803 columns:
  the price, the occurrence total with its 400 cells, and the amount total with its 400 cells.
* The price file's transaction count equals the occurrence total on all 2,404 days that have a price. A
  shift of even one day would break that on almost every row, so the price file and the chainlet files
  are aligned on the same calendar date, although their dates were built in two different ways.
* The only empty cells are the prices of 2012-12-31 and 2016-12-31. Both days have chainlet rows, so the
  gaps are in the price file alone.
* September to December 2017 is complete in every column.
* Chainlet history from 2009 to May 2011 is left out. A day with no price can be neither a target nor a
  lag, so that history has no use here.

## 3. Feature engineering

## 4. Split and scaling

## 5. Model selection

## 6. Final evaluation on December 2017

## 7. Explaining the model with SHAP

## 8. Predictions file

## 9. Compliance audit